In [ ]:
"""
I-AutoRec for MovieLens 10M  —  Optimized Implementation
==========================================================
Paper: "AutoRec: Autoencoders Meet Collaborative Filtering"
Sedhain et al., WWW 2015   |   Target RMSE: 0.782

Key differences vs U-AutoRec:
  - Item-based autoencoder (each item's user-rating vector is input)
  - ML-10M is ~28x larger than ML-1M  →  memory-efficient sparse handling
  - n_users ~ 69,878  vs  n_items ~ 10,677  (items << users for ML-10M)
    So item vectors are much denser → I-AutoRec works better here
  - lambda_reg best value differs from U-AutoRec

Architecture:
  h(r; θ) = f( W · g(V·r + μ) + b )
    g(·) = Sigmoid   (hidden activation)
    f(·) = Identity  (output activation)
    input dimension = n_users  (each item rated by subset of users)

ML-10M format: ratings.dat  →  UserID::MovieID::Rating::Timestamp
  Ratings: 10,000,054   Users: 69,878   Movies: 10,677

Kaggle-ready:
  - Config dataclass (no argparse — no kernel conflict)
  - num_workers=0
  - Auto-discovers ratings.dat
  - Best model only saved per fold
  - Memory-efficient: sparse matrix → dense only per batch
"""

import os
import gc
import time
import random
import dataclasses as _dc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from dataclasses import dataclass
import logging
import scipy.sparse as sp

# ──────────────────────────────────────────────────────────────
# Logging
# ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────
# Config  ← only thing you need to edit
# ──────────────────────────────────────────────────────────────
@dataclass
class Config:
    # ── Paths ─────────────────────────────────────────────────
    # Folder containing ML-10M data (ratings.dat found automatically)
    dataset_root:   str   = "/kaggle/input/datasets/priyanshuunayak/ml-10m-rating"
    checkpoint_dir: str   = "/kaggle/working/i_autorec_10m"

    # ── Model  (paper best) ───────────────────────────────────
    hidden_units:   int   = 500       # paper Figure 2: best at k=500

    # ── Training  (paper exact settings) ─────────────────────
    epochs:         int   = 200
    # ML-10M has ~10K items — batch of 64 items is stable
    # (each item vector has ~70K dims, so keep batch small)
    batch_size:     int   = 64
    lr:             float = 0.001     # RProp initial step size
    # I-AutoRec lambda: paper tunes {0.001,0.01,0.1,1,100,1000}
    # Best for I-AutoRec ML-10M is typically 1.0
    lambda_reg:     float = 1.0

    # ── Early stopping ────────────────────────────────────────
    early_stop:     int   = 20        # patience on val RMSE
    log_every:      int   = 10        # print every N epochs

    # ── System ────────────────────────────────────────────────
    seed:           int   = 42
    num_workers:    int   = 0         # must be 0 on Kaggle
    device:         str   = "auto"    # "auto" | "cuda" | "cpu"


# ──────────────────────────────────────────────────────────────
# Utilities
# ──────────────────────────────────────────────────────────────
def cfg_to_dict(cfg):
    try:
        return _dc.asdict(cfg)
    except TypeError:
        return vars(cfg)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False


# ──────────────────────────────────────────────────────────────
# Auto-discover ratings.dat
# ──────────────────────────────────────────────────────────────
def find_ratings_file(root: str) -> str:
    logger.info(f"Searching for ratings.dat under: {root}")
    for dirpath, _, files in os.walk(root):
        if "ratings.dat" in files:
            path = os.path.join(dirpath, "ratings.dat")
            logger.info(f"  Found: {path}")
            return path
    all_files = [
        os.path.join(d, f)
        for d, _, fs in os.walk(root)
        for f in fs
    ]
    listing = "\n  ".join(all_files) or "(empty)"
    raise FileNotFoundError(
        f"ratings.dat not found under '{root}'.\n"
        f"Files present:\n  {listing}\n"
        f"→ Fix 'dataset_root' in Config."
    )


# ──────────────────────────────────────────────────────────────
# Data Loading
# ──────────────────────────────────────────────────────────────
def load_ml10m(path: str):
    """
    Parse ML-10M ratings.dat → zero-indexed numpy array (N, 3).
    Columns: [user_idx, item_idx, rating]

    ML-10M format: UserID::MovieID::Rating::Timestamp
    Ratings are floats: 0.5 to 5.0 in steps of 0.5
    """
    logger.info(f"Loading ML-10M from: {path}")
    logger.info("  (this may take ~30s for 10M ratings...)")

    raw_u, raw_i, raw_r = [], [], []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("::")
            if len(parts) < 3:
                continue
            raw_u.append(int(parts[0]))
            raw_i.append(int(parts[1]))
            raw_r.append(float(parts[2]))

    raw_u = np.array(raw_u, dtype=np.int32)
    raw_i = np.array(raw_i, dtype=np.int32)
    raw_r = np.array(raw_r, dtype=np.float32)

    # Zero-index users and items
    unique_u = np.unique(raw_u)
    unique_i = np.unique(raw_i)
    u2i = {u: i for i, u in enumerate(unique_u)}
    i2i = {it: i for i, it in enumerate(unique_i)}

    u_idx = np.array([u2i[u] for u in raw_u], dtype=np.int32)
    i_idx = np.array([i2i[i] for i in raw_i], dtype=np.int32)

    n_users = len(unique_u)
    n_items = len(unique_i)

    data = np.column_stack([u_idx, i_idx, raw_r]).astype(np.float32)

    logger.info(f"  Users={n_users:,}  Items={n_items:,}  Ratings={len(data):,}")
    logger.info(f"  Rating range: [{raw_r.min():.1f}, {raw_r.max():.1f}]")
    return data, n_users, n_items


# ──────────────────────────────────────────────────────────────
# Splitting
# ──────────────────────────────────────────────────────────────
def split_90_10(data, seed):
    """90% trainval / 10% test — paper exact procedure."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(data))
    cut = int(len(data) * 0.9)
    return data[idx[:cut]], data[idx[cut:]]


def split_val(trainval, seed):
    """10% of trainval → validation."""
    rng = np.random.default_rng(seed + 1000)
    idx = rng.permutation(len(trainval))
    cut = int(len(trainval) * 0.9)
    return trainval[idx[:cut]], trainval[idx[cut:]]


# ──────────────────────────────────────────────────────────────
# Rating Matrix — SPARSE for ML-10M  (memory critical)
# ──────────────────────────────────────────────────────────────
def make_sparse_matrix(data, n_users, n_items):
    """
    Build CSR sparse matrix (n_items × n_users) from train data.
    Items are rows, users are columns — I-AutoRec input is per-item.

    ML-10M dense matrix would be:
      69878 users × 10677 items × 4 bytes = ~2.8 GB  ← too large
    Sparse CSR uses ~150 MB instead.
    """
    u_idx = data[:, 0].astype(np.int32)
    i_idx = data[:, 1].astype(np.int32)
    vals  = data[:, 2].astype(np.float32)

    # shape: (n_items, n_users)  — each row is one item's user rating vector
    R_sparse = sp.csr_matrix(
        (vals, (i_idx, u_idx)),
        shape=(n_items, n_users),
        dtype=np.float32,
    )
    return R_sparse


# ──────────────────────────────────────────────────────────────
# Dataset — converts sparse rows to dense on the fly
# ──────────────────────────────────────────────────────────────
class ItemRatingDataset(Dataset):
    """
    Each sample = one item's partially observed user rating vector.
    Input to I-AutoRec: r^(i) = (R_1i, R_2i, ..., R_mi) ∈ R^m

    Uses sparse matrix to avoid storing full dense matrix in RAM.
    Converts to dense only for the requested batch.
    """
    def __init__(self, R_sparse: sp.csr_matrix):
        self.R = R_sparse                  # (n_items, n_users)  sparse
        self.n_items = R_sparse.shape[0]

    def __len__(self):
        return self.n_items

    def __getitem__(self, idx):
        # Convert single sparse row to dense
        row  = np.asarray(self.R[idx].todense(), dtype=np.float32).squeeze()
        mask = (row != 0).astype(np.float32)
        return row, mask


# ──────────────────────────────────────────────────────────────
# I-AutoRec Model
# ──────────────────────────────────────────────────────────────
class IAutoRec(nn.Module):
    """
    Item-based AutoRec.

        h(r; θ) = f( W · g(V·r + μ) + b )

    Input  : r^(i) ∈ R^m  — item i's partially observed user ratings
             m = n_users (~69,878 for ML-10M)
    Hidden : k = 500  (Sigmoid activation)
    Output : R^m       (Identity activation)

    g(·) = Sigmoid   ← hidden activation
    f(·) = Identity  ← output activation  (best per paper Table 1b)

    Parameters: 2 * n_users * k + k + n_users
    """
    def __init__(self, n_users: int, k: int = 500):
        super().__init__()
        self.encoder = nn.Linear(n_users, k)       # V, μ
        self.decoder = nn.Linear(k, n_users)       # W, b
        self.g = nn.Sigmoid()
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.encoder.weight)
        nn.init.xavier_uniform_(self.decoder.weight)
        nn.init.zeros_(self.encoder.bias)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, r):
        """
        r      : (batch, n_users)  — partially observed item vectors
        returns: (batch, n_users)  — reconstructed user ratings for items
        """
        return self.decoder(self.g(self.encoder(r)))

    def l2_penalty(self):
        """Global L2 on weight matrices only (not biases) — paper Eq.2."""
        return (
            self.encoder.weight.norm(p="fro") ** 2 +
            self.decoder.weight.norm(p="fro") ** 2
        )


# ──────────────────────────────────────────────────────────────
# Loss — masked MSE (observed ratings only)
# ──────────────────────────────────────────────────────────────
def masked_mse(pred, target, mask):
    """
    MSE over OBSERVED ratings only.
    Implements  ||r^(i) - h(r^(i))||²_O  from paper Eq.2.
    """
    diff  = (pred - target) * mask
    n_obs = mask.sum()
    if n_obs == 0:
        return torch.tensor(0.0, device=pred.device, requires_grad=True)
    return (diff ** 2).sum() / n_obs


# ──────────────────────────────────────────────────────────────
# Evaluation — global RMSE over all test pairs
# ──────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_rmse(model, test_pairs, R_sparse, device, item_chunk=256):
    """
    Compute RMSE on (user, item, true_rating) test pairs.

    Strategy for ML-10M memory efficiency:
      - Process items in chunks of `item_chunk`
      - For each chunk, convert sparse rows to dense, run model,
        collect predictions for test pairs belonging to those items
      - Never materialise the full dense matrix

    Predictions clipped to [0.5, 5.0] — ML-10M rating range.
    """
    model.eval()
    n_items = R_sparse.shape[0]

    # Group test pairs by item for efficient lookup
    # item_to_pairs[item_idx] = list of (user_idx, true_rating)
    from collections import defaultdict
    item_to_pairs = defaultdict(list)
    for u, i, r in test_pairs:
        item_to_pairs[int(i)].append((int(u), float(r)))

    preds_all   = []
    targets_all = []

    for start in range(0, n_items, item_chunk):
        end   = min(start + item_chunk, n_items)
        chunk = R_sparse[start:end]                     # sparse slice

        # Dense conversion only for this chunk
        batch = torch.from_numpy(
            np.asarray(chunk.todense(), dtype=np.float32)
        ).to(device)                                    # (chunk_size, n_users)

        recon = model(batch).cpu().numpy()              # (chunk_size, n_users)

        for local_i, global_i in enumerate(range(start, end)):
            if global_i not in item_to_pairs:
                continue
            for u_idx, true_r in item_to_pairs[global_i]:
                pred = float(np.clip(recon[local_i, u_idx], 0.5, 5.0))
                preds_all.append(pred)
                targets_all.append(true_r)

    if len(preds_all) == 0:
        return float("nan")

    preds   = np.array(preds_all,   dtype=np.float64)
    targets = np.array(targets_all, dtype=np.float64)
    return float(np.sqrt(np.mean((preds - targets) ** 2)))


# ──────────────────────────────────────────────────────────────
# One Training Epoch
# ──────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, lambda_reg, device):
    """
    One pass over all items.
    Loss = masked_MSE(pred, R, mask) + (lambda/2) * (||V||²_F + ||W||²_F)
    """
    model.train()
    running_loss = 0.0

    bar = tqdm(loader, desc="  train", leave=False, ncols=90)
    for R_batch, mask_batch in bar:
        R_batch    = R_batch.to(device, non_blocking=True)
        mask_batch = mask_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(R_batch)

        rec_loss = masked_mse(pred, R_batch, mask_batch)
        l2_loss  = (lambda_reg / 2.0) * model.l2_penalty()
        loss     = rec_loss + l2_loss

        loss.backward()
        optimizer.step()

        running_loss += rec_loss.item()
        bar.set_postfix(rec=f"{rec_loss.item():.4f}")

    return running_loss / max(len(loader), 1)


# ──────────────────────────────────────────────────────────────
# Single Fold
# ──────────────────────────────────────────────────────────────
def run_fold(fold_id, train_data, val_data, test_data,
             n_users, n_items, cfg, device):

    logger.info(f"\n{'='*64}")
    logger.info(f"  FOLD {fold_id}/5  "
                f"train={len(train_data):,}  "
                f"val={len(val_data):,}  "
                f"test={len(test_data):,}")
    logger.info(f"{'='*64}")

    # Build sparse item-user matrix from train data ONLY
    logger.info("  Building sparse train matrix (items × users)...")
    R_sparse = make_sparse_matrix(train_data, n_users, n_items)
    logger.info(f"  Sparse matrix: {R_sparse.shape}  "
                f"nnz={R_sparse.nnz:,}  "
                f"density={100*R_sparse.nnz/(n_items*n_users):.3f}%")

    dataset = ItemRatingDataset(R_sparse)
    loader  = DataLoader(
        dataset,
        batch_size  = cfg.batch_size,
        shuffle     = True,
        num_workers = cfg.num_workers,
        pin_memory  = (device.type == "cuda"),
        drop_last   = False,
    )

    # Model input dim = n_users (item vector length)
    model = IAutoRec(n_users=n_users, k=cfg.hidden_units).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    logger.info(f"  IAutoRec  input={n_users:,}  k={cfg.hidden_units}  "
                f"params={n_params:,}")

    # RProp — exact optimiser from paper
    optimizer = optim.Rprop(
        model.parameters(),
        lr         = cfg.lr,
        etas       = (0.5, 1.2),
        step_sizes = (1e-6, 50),
    )

    os.makedirs(cfg.checkpoint_dir, exist_ok=True)
    best_path     = os.path.join(cfg.checkpoint_dir, f"fold{fold_id}_best.pt")
    best_val_rmse = float("inf")
    best_epoch    = 0
    patience      = 0

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()

        train_loss = train_epoch(model, loader, optimizer, cfg.lambda_reg, device)
        val_rmse   = evaluate_rmse(model, val_data, R_sparse, device)
        elapsed    = time.time() - t0

        if epoch % cfg.log_every == 0 or epoch == 1:
            logger.info(
                f"  Ep {epoch:4d}/{cfg.epochs}  "
                f"loss={train_loss:.5f}  "
                f"val_RMSE={val_rmse:.4f}  "
                f"best={best_val_rmse:.4f}  "
                f"({elapsed:.1f}s)"
            )

        # Save best model only
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch    = epoch
            patience      = 0
            torch.save({
                "fold":     fold_id,
                "epoch":    epoch,
                "model":    model.state_dict(),
                "val_rmse": val_rmse,
                "config":   cfg_to_dict(cfg),
            }, best_path)
            logger.info(
                f"  ✓ New best  val_RMSE={val_rmse:.4f}"
                f"  → {os.path.basename(best_path)}"
            )
        else:
            patience += 1

        if cfg.early_stop > 0 and patience >= cfg.early_stop:
            logger.info(f"  Early stop at epoch {epoch} "
                        f"(patience={cfg.early_stop})")
            break

    # Final test RMSE with best model
    ckpt = torch.load(best_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt["model"])
    test_rmse = evaluate_rmse(model, test_data, R_sparse, device)

    logger.info(f"\n  Fold {fold_id} done  →  "
                f"best_val={best_val_rmse:.4f} (ep {best_epoch})  "
                f"test_RMSE={test_rmse:.4f}")

    # Free memory before next fold
    del R_sparse, dataset, loader
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return test_rmse


# ──────────────────────────────────────────────────────────────
# 5-Fold Cross Validation
# ──────────────────────────────────────────────────────────────
def run_cv(all_data, n_users, n_items, cfg, device):
    """
    Paper procedure (Section 3):
      - 90/10 random split, repeated 5 times
      - 10% of train held out as validation
      - Report mean ± std RMSE
    """
    os.makedirs(cfg.checkpoint_dir, exist_ok=True)

    logger.info("\n" + "="*64)
    logger.info("  I-AutoRec ML-10M  —  5-Fold Cross Validation")
    logger.info("="*64)

    fold_rmses = []

    for fold in range(1, 6):
        set_seed(cfg.seed + fold)

        trainval, test = split_90_10(all_data, seed=cfg.seed + fold)
        train, val     = split_val(trainval,   seed=cfg.seed + fold)

        rmse = run_fold(
            fold_id    = fold,
            train_data = train,
            val_data   = val,
            test_data  = test,
            n_users    = n_users,
            n_items    = n_items,
            cfg        = cfg,
            device     = device,
        )
        fold_rmses.append(rmse)

    avg = float(np.mean(fold_rmses))
    std = float(np.std(fold_rmses))

    logger.info("\n" + "="*64)
    logger.info("  FINAL RESULTS")
    logger.info("="*64)
    for i, r in enumerate(fold_rmses):
        logger.info(f"  Fold {i+1}:  RMSE = {r:.4f}")
    logger.info(f"\n  Mean RMSE : {avg:.4f}  ±  {std:.4f}")
    logger.info(f"  Paper     : 0.782  (I-AutoRec ML-10M)")
    logger.info("="*64)

    # Save summary
    out = os.path.join(cfg.checkpoint_dir, "results.txt")
    with open(out, "w") as f:
        f.write("I-AutoRec ML-10M  —  Replication\n")
        f.write("="*40 + "\n\n")
        for i, r in enumerate(fold_rmses):
            f.write(f"Fold {i+1}: {r:.4f}\n")
        f.write(f"\nMean : {avg:.4f} ± {std:.4f}\n")
        f.write(f"Paper: 0.782\n\n")
        f.write(f"Config: {cfg_to_dict(cfg)}\n")
    logger.info(f"Summary → {out}")

    return avg, std, fold_rmses


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main():
    cfg = Config()
    set_seed(cfg.seed)

    # Device
    if cfg.device == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(cfg.device)

    logger.info(f"Device : {device}")
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(0)
        logger.info(f"  GPU  : {props.name}")
        logger.info(f"  VRAM : {props.total_memory / 1e9:.1f} GB")

    # Locate data
    ratings_path = find_ratings_file(cfg.dataset_root)

    # Load — takes ~30s for 10M rows
    all_data, n_users, n_items = load_ml10m(ratings_path)

    # Config summary
    logger.info(f"\n--- Config ---")
    logger.info(f"  model        : I-AutoRec  (item-based)")
    logger.info(f"  hidden_units : {cfg.hidden_units}")
    logger.info(f"  lambda_reg   : {cfg.lambda_reg}  ← key param")
    logger.info(f"  epochs       : {cfg.epochs}")
    logger.info(f"  batch_size   : {cfg.batch_size}  (items per batch)")
    logger.info(f"  early_stop   : {cfg.early_stop}")
    logger.info(f"  optimizer    : RProp (lr={cfg.lr})")
    logger.info(f"  activation   : g=Sigmoid, f=Identity  (Table 1b best)")
    logger.info(f"  input_dim    : {n_users:,} (user ratings per item)")
    logger.info(f"  num_workers  : {cfg.num_workers}")

    run_cv(all_data, n_users, n_items, cfg, device)


if __name__ == "__main__":
    main()

03:03:32 | INFO | Device : cuda
03:03:32 | INFO |   GPU  : Tesla T4
03:03:32 | INFO |   VRAM : 15.6 GB
03:03:32 | INFO | Searching for ratings.dat under: /kaggle/input/datasets/priyanshuunayak/ml-10m-rating
03:03:32 | INFO |   Found: /kaggle/input/datasets/priyanshuunayak/ml-10m-rating/ratings.dat
03:03:32 | INFO | Loading ML-10M from: /kaggle/input/datasets/priyanshuunayak/ml-10m-rating/ratings.dat
03:03:32 | INFO |   (this may take ~30s for 10M ratings...)
03:03:47 | INFO |   Users=69,878  Items=10,677  Ratings=10,000,054
03:03:47 | INFO |   Rating range: [0.5, 5.0]
03:03:47 | INFO | 
--- Config ---
03:03:47 | INFO |   model        : I-AutoRec  (item-based)
03:03:47 | INFO |   hidden_units : 500
03:03:47 | INFO |   lambda_reg   : 1.0  ← key param
03:03:47 | INFO |   epochs       : 200
03:03:47 | INFO |   batch_size   : 64  (items per batch)
03:03:47 | INFO |   early_stop   : 20
03:03:47 | INFO |   optimizer    : RProp (lr=0.001)
03:03:47 | INFO |   activation   : g=Sigmoid, f=Identit